<div style="background: linear-gradient(135deg, #0a0a0a 0%, #1a1a2e 35%, #0d3b5e 100%);padding: 60px 40px; border-radius: 18px; text-align: center; margin-bottom: 10px; border: 1px solid #4a9eff44;"><h1 style="color: #4a9eff; font-family: 'Courier New', monospace; font-size: 2.7em;letter-spacing: 4px; margin: 0 0 12px 0; text-shadow: 0 0 40px #4a9eff55;">✅ ALIŞTIRMA ÇÖZÜMLERİ</h1><h2 style="color: #a8dadc; font-family: 'Courier New', monospace;font-size: 1.3em; margin: 0 0 18px 0; font-weight: 300;">8. Hafta — Geometrik Dönüşümler</h2><p style="color: #6b7280; font-family: 'Courier New', monospace; font-size: 0.85em; margin: 0;">SSIM/PSNR &bull; Belge Tarayıcı &bull; Laplacian Piramidi &bull; Video Stabilizasyon &bull; Augmentation Pipeline</p></div>

## 📋 Bu Notebook Hakkında

Bu notebook, 8. hafta alıştırmalarının **adım adım, detaylı çözümlerini** içermektedir. Her soru için:

1. Problemin arka planı ve neden önemli olduğu
2. Matematiksel temel
3. Adım adım uygulama
4. Çalışan tam kod
5. Sonuçların yorumlanması

> **Not:** Kodlar OpenCV gerektiren bölümlerde gerçek görüntü dosyası olmadan da çalışabilmesi için sentetik test görüntüleri kullanılmaktadır.

---
# 🔬 SORU 1: SSIM Tabanlı Kalite Değerlendirici

## 1.1 Problem Nedir?

Farklı interpolasyon yöntemlerinin kalitesini **gözle değil, sayısal olarak** karşılaştırmak istiyoruz. Bunun için üç farklı metrik kullanacağız: **MSE**, **PSNR** ve **SSIM**.

## 1.2 Metrikler — Matematiksel Temel

### MSE (Mean Squared Error — Ortalama Karesel Hata)

$$MSE = \frac{1}{MN} \sum_{i=0}^{M-1} \sum_{j=0}^{N-1} [I(i,j) - K(i,j)]^2$$

- $I$: orijinal görüntü, $K$: karşılaştırılan görüntü
- $M \times N$: piksel sayısı
- **MSE = 0** → mükemmel eşleşme, **MSE büyük** → kötü kalite
- Birimi: piksel² — yorumlanması zor

### PSNR (Peak Signal-to-Noise Ratio — Tepe Sinyal Gürültü Oranı)

$$PSNR = 20 \cdot \log_{10}\left(\frac{MAX_I}{\sqrt{MSE}}\right) = 10 \cdot \log_{10}\left(\frac{MAX_I^2}{MSE}\right)$$

- $MAX_I = 255$ (uint8 görüntüler için)
- Birimi: **dB (desibel)** — logaritmik ölçek
- **>40 dB**: çok iyi, **30-40 dB**: iyi, **<30 dB**: görünür bozulma

### SSIM (Structural Similarity Index)

$$SSIM(x,y) = \frac{(2\mu_x\mu_y + C_1)(2\sigma_{xy} + C_2)}{(\mu_x^2 + \mu_y^2 + C_1)(\sigma_x^2 + \sigma_y^2 + C_2)}$$

- $\mu$: ortalama (parlaklık), $\sigma^2$: varyans (kontrast), $\sigma_{xy}$: kovaryans (yapı)
- **SSIM = 1.0**: mükemmel, **SSIM = 0**: hiç benzerlik yok
- İnsanın görsel algısını PSNR'dan çok daha iyi modellemektedir

<div style="background: #2d1800; border-left: 5px solid #f4a261; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #f4a261; font-size: 1.08em;">📜 Tarihçe</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>SSIM'in Doğuşu — 2004, Wang ve Bovik:</b><br><br>PSNR, 1970'lerden beri kullanılıyordu ama ciddi bir sorunu vardı: insan gözünün beğendiği görüntülerle PSNR'ın yüksek bulduğu görüntüler her zaman örtüşmüyordu. Hafif bulanık ama doğal görünen bir görüntü keskin ama artefaktlı bir görüntüden düşük PSNR alabiliyordu.<br><br>2004'te Zhou Wang ve Alan Bovik, "Image Quality Assessment: From Error Visibility to Structural Similarity" başlıklı makaleyi yayımladı. Temel fikir: insan görsel sistemi yapısal bilgiyi (kenarlar, dokular, şekiller) piksel piksel farktan çok daha önemser. SSIM bu üç bileşeni (parlaklık, kontrast, yapı) ayrı ayrı ölçüp çarpar.<br><br>Bu makale bilgisayar görüsü alanında <b>en çok atıf alan makaleler</b> arasına girdi. Netflix, YouTube, YouTube: tüm büyük video platformları SSIM (veya türevleri) kullanarak sıkıştırma kalitesini ölçer. VMAF (Netflix'in geliştirdiği metrik) SSIM'in doğrudan mirasçısıdır.</span></div>

## 1.3 Adım Adım Çözüm

<div style="background: #0d2200; border-left: 5px solid #90be6d; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #90be6d; font-size: 1.08em;">📍 Adım 1 </strong><br><br><span style="color: #cdd6f4; line-height: 1.9;">Gerekli kütüphaneleri import et, test görüntüsü oluştur (veya gerçek görüntü yükle).</span></div>

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ─── Test görüntüsü: karmaşık desen (daha iyi interpolasyon farklılığı gösterir) ───
np.random.seed(42)
orijinal = np.zeros((256, 256, 3), dtype=np.uint8)

# Geometrik şekiller, metin ve gürültü: interpolasyon farklarını net gösterir
cv.circle(orijinal, (128, 128), 80, (200, 50, 50), -1)
cv.rectangle(orijinal, (20, 20), (100, 100), (50, 200, 50), -1)
cv.line(orijinal, (0, 0), (255, 255), (255, 255, 0), 2)
cv.putText(orijinal, "TEST", (60, 220), cv.FONT_HERSHEY_DUPLEX, 1.5, (255,255,255), 2)

# İnce çizgiler — interpolasyon farklarının en belirgin göründüğü yer
for i in range(0, 256, 4):
    orijinal[i, :] = np.clip(orijinal[i, :].astype(int) + 30, 0, 255)

print(f"Orijinal boyut: {orijinal.shape}")
plt.figure(figsize=(4, 4))
plt.imshow(cv.cvtColor(orijinal, cv.COLOR_BGR2RGB))
plt.title("Test Görüntüsü")
plt.axis("off")
plt.tight_layout()
plt.show()

<div style="background: #0d2200; border-left: 5px solid #90be6d; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #90be6d; font-size: 1.08em;">📍 Adım 2 </strong><br><br><span style="color: #cdd6f4; line-height: 1.9;">MSE ve PSNR hesaplama fonksiyonlarını tanımla.</span></div>

In [ ]:
def hesapla_mse(img1, img2):
    """
    MSE = Ortalama Karesel Hata
    İki görüntü arasındaki piksel farkının karesinin ortalaması.
    """
    # uint8'i int16'ya çevir: çıkarma işleminde negatif taşma olmaz
    fark = img1.astype(np.float64) - img2.astype(np.float64)
    mse = np.mean(fark ** 2)
    return mse


def hesapla_psnr(img1, img2):
    """
    PSNR = 10 * log10(MAX² / MSE)
    Birimi dB. Yüksek = iyi kalite.
    """
    mse = hesapla_mse(img1, img2)
    if mse == 0:
        return float('inf')  # mükemmel eşleşme
    max_piksel = 255.0
    psnr = 10 * np.log10((max_piksel ** 2) / mse)
    return psnr


def hesapla_ssim(img1, img2):
    """
    SSIM: Wang & Bovik (2004)
    Parlaklık × Kontrast × Yapı bileşenlerinin çarpımı.
    
    Pencere tabanlı hesaplama: her pikselin çevresindeki 11×11 bölge
    için ayrı ayrı SSIM hesaplanır, sonra ortalama alınır.
    """
    # Sabitleri (kararlılık için küçük değerler)
    C1 = (0.01 * 255) ** 2   # L=255, k1=0.01
    C2 = (0.03 * 255) ** 2   # L=255, k2=0.03

    img1_f = img1.astype(np.float64)
    img2_f = img2.astype(np.float64)

    # Gaussian ağırlıklı yerel ortalama (11×11 pencere, σ=1.5)
    kernel_size = 11
    sigma = 1.5
    kernel = cv.getGaussianKernel(kernel_size, sigma)
    window = np.outer(kernel, kernel.T)

    mu1 = cv.filter2D(img1_f, -1, window)[5:-5, 5:-5]   # yerel ortalama
    mu2 = cv.filter2D(img2_f, -1, window)[5:-5, 5:-5]

    mu1_sq = mu1 ** 2
    mu2_sq = mu2 ** 2
    mu1_mu2 = mu1 * mu2

    sigma1_sq = cv.filter2D(img1_f ** 2, -1, window)[5:-5, 5:-5] - mu1_sq   # yerel varyans
    sigma2_sq = cv.filter2D(img2_f ** 2, -1, window)[5:-5, 5:-5] - mu2_sq
    sigma12   = cv.filter2D(img1_f * img2_f, -1, window)[5:-5, 5:-5] - mu1_mu2  # yerel kovaryans

    # SSIM formülü
    pay    = (2 * mu1_mu2 + C1) * (2 * sigma12 + C2)
    payda  = (mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2)
    ssim_map = pay / payda

    return float(np.mean(ssim_map))


# Sağlama: özdeş görüntüler için beklenen değerler
test_a = np.ones((100, 100, 3), dtype=np.uint8) * 128
test_b = np.ones((100, 100, 3), dtype=np.uint8) * 128
print(f"Özdeş görüntüler → MSE={hesapla_mse(test_a,test_b):.2f}, "
      f"PSNR={hesapla_psnr(test_a,test_b)}, "
      f"SSIM={hesapla_ssim(test_a[:,:,0], test_b[:,:,0]):.4f}")

<div style="background: #0d2200; border-left: 5px solid #90be6d; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #90be6d; font-size: 1.08em;">📍 Adım 3 </strong><br><br><span style="color: #cdd6f4; line-height: 1.9;">Her interpolasyon yöntemi için: küçült → büyüt → karşılaştır.</span></div>

In [ ]:
YONTEMLER = {
    "INTER_NEAREST":  cv.INTER_NEAREST,
    "INTER_LINEAR":   cv.INTER_LINEAR,
    "INTER_CUBIC":    cv.INTER_CUBIC,
    "INTER_LANCZOS4": cv.INTER_LANCZOS4,
    "INTER_AREA":     cv.INTER_AREA,
}

SONUCLAR = {}

for isim, yontem in YONTEMLER.items():
    h, w = orijinal.shape[:2]

    # Adım 1: %50 küçült
    kucuk = cv.resize(orijinal, (w//2, h//2), interpolation=yontem)

    # Adım 2: Orijinal boyuta geri büyüt
    # ÖNEMLİ: Küçültme için INTER_AREA en iyi; büyütme için test ettiğimiz yöntemi kullan
    geri = cv.resize(kucuk, (w, h), interpolation=yontem)

    # Adım 3: Metrik hesapla (gri kanalda — SSIM tek kanal bekler)
    mse_val  = hesapla_mse(orijinal, geri)
    psnr_val = hesapla_psnr(orijinal, geri)
    ssim_val = hesapla_ssim(
        cv.cvtColor(orijinal, cv.COLOR_BGR2GRAY),
        cv.cvtColor(geri,     cv.COLOR_BGR2GRAY)
    )

    SONUCLAR[isim] = {
        "mse":   mse_val,
        "psnr":  psnr_val,
        "ssim":  ssim_val,
        "geri":  geri
    }
    print(f"{isim:<20} MSE={mse_val:7.2f}  PSNR={psnr_val:6.2f} dB  SSIM={ssim_val:.4f}")

<div style="background: #0d2200; border-left: 5px solid #90be6d; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #90be6d; font-size: 1.08em;">📍 Adım 4 </strong><br><br><span style="color: #cdd6f4; line-height: 1.9;">Sonuçları tablo ve görsel olarak karşılaştır.</span></div>

In [ ]:
# ─── Görsel karşılaştırma ───
fig = plt.figure(figsize=(16, 10))
fig.suptitle("İnterpolasyon Kalite Karşılaştırması\n(Küçüt %50 → Geri Büyüt, Orijinalle Karşılaştır)",
             fontsize=13, fontweight="bold")

n = len(YONTEMLER)
for idx, (isim, veri) in enumerate(SONUCLAR.items()):
    # Yeniden yapılandırılmış görüntü
    ax = fig.add_subplot(3, n+1, idx+2)
    ax.imshow(cv.cvtColor(veri["geri"], cv.COLOR_BGR2RGB))
    ax.set_title(f"{isim}\nSSIM={veri['ssim']:.3f}\nPSNR={veri['psnr']:.1f}dB", fontsize=7)
    ax.axis("off")

    # Fark görüntüsü (×5 büyütülmüş)
    ax2 = fig.add_subplot(3, n+1, n+1+idx+2)
    fark = cv.absdiff(orijinal, veri["geri"])
    fark_rgb = cv.cvtColor(fark, cv.COLOR_BGR2RGB)
    ax2.imshow(np.clip(fark_rgb * 5, 0, 255).astype(np.uint8))
    ax2.set_title(f"Fark ×5\nMSE={veri['mse']:.1f}", fontsize=7)
    ax2.axis("off")

# Orijinal
ax_orig = fig.add_subplot(3, n+1, 1)
ax_orig.imshow(cv.cvtColor(orijinal, cv.COLOR_BGR2RGB))
ax_orig.set_title("Orijinal", fontsize=9, fontweight="bold")
ax_orig.axis("off")

ax_orig2 = fig.add_subplot(3, n+1, n+2)
ax_orig2.imshow(np.zeros_like(orijinal))
ax_orig2.set_title("(Referans fark\n= sıfır)", fontsize=7)
ax_orig2.axis("off")

# Metrik barları
ax_bar = fig.add_subplot(3, 1, 3)
isimleri = list(SONUCLAR.keys())
ssim_vals = [SONUCLAR[i]["ssim"] for i in isimleri]
renkler = ["#e94560" if v < 0.7 else "#f9c74f" if v < 0.85 else "#90be6d" for v in ssim_vals]
bars = ax_bar.bar(isimleri, ssim_vals, color=renkler, edgecolor="white", linewidth=0.5)
ax_bar.set_ylabel("SSIM (1.0 = mükemmel)", fontsize=9)
ax_bar.set_title("SSIM Karşılaştırması (Yüksek = İyi)", fontsize=10)
ax_bar.set_ylim(0, 1.05)
ax_bar.axhline(1.0, color="white", linestyle="--", alpha=0.3, label="Mükemmel")
for bar, val in zip(bars, ssim_vals):
    ax_bar.text(bar.get_x() + bar.get_width()/2, val + 0.01,
                f"{val:.3f}", ha="center", va="bottom", fontsize=8, color="white")
plt.tight_layout()
plt.show()

## 1.4 Sonuçların Yorumu

<div style="background: #0d1e3d; border-left: 5px solid #4a9eff; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #4a9eff; font-size: 1.08em;">💡 Anahtar Fikir</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>Neden PSNR yüksek olmasına rağmen görüntü kötü görünebilir?</b><br><br>PSNR yalnızca <b>piksel piksel mutlak fark</b>ı ölçer. Görüntü yarım piksel sola kaydırılsa bile PSNR çok düşer — ama insan gözü bu kaymayı fark etmez.<br><br>Tersine: küçük ama yapısal bir bozulma (bir kenarın bulanıklaşması) MSE'yi az artırır ama gözle hemen fark edilir. SSIM bu yüzden üç bileşeni ayrı ölçer: <b>parlaklık × kontrast × yapı</b>.<br><br>Netflix'in VMAF sistemi SSIM'in bu fikrini daha da genişletir: insan görsel korteksinin farklı frekans bandlarına verdiği yanıtı simüle eden 6 farklı metriği makine öğrenmesiyle birleştirir.</span></div>

---
# 📄 SORU 2: Otomatik Belge Tarayıcı

## 2.1 Problem ve Motivasyon

Bir kağıdı fotoğrafladığınızda perspektif bozukluğu oluşur: kağıt dikdörtgen olmasına rağmen fotoğrafta yamuk görünür. Bu soruyu çözmek, CamScanner, Microsoft Office Lens, Adobe Scan gibi uygulamaların temelini anlamak demektir.

## 2.2 Affine vs Perspektif Dönüşüm — Temel Fark

```
Affine Dönüşüm:              Perspektif Dönüşüm:

A ─────── B                  A ──────────── B
│         │                   \              /
│         │        →           \            /
│         │                     C ──────── D
D ─────── C

Paralel çizgiler KORUNUR     Paralel çizgiler KORUNMAYABLIR
3 nokta yeterli              4 nokta gerekli
2×3 matris                   3×3 matris
```

**Belge tarama neden perspektif ister?**
Kameraya açılı tutulan kağıtta paralel kenarlar perspektifle kapanır — Affine bu durumu düzeltemez, perspektif dönüşüm gerekir.

<div style="background: #2d1800; border-left: 5px solid #f4a261; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #f4a261; font-size: 1.08em;">📜 Tarihçe</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>Perspektif Dönüşümün Tarihi — Rönesans'tan Bilgisayara:</b><br><br>Perspektif geometrisini sistematik olarak ilk inceleyen Filippo Brunelleschi'dir (1420'ler). Daha sonra Alberti ve Leonardo da Vinci bunu resim sanatına uyguladı.<br><br>19. yüzyılda fotoğrafın icadıyla lens distorsiyon düzeltme problemi ortaya çıktı. Fotogrametri (fotoğraftan ölçüm alma) bilim dalı bu ihtiyaçtan doğdu ve bugün uydu görüntülerinden 3B harita oluşturmada kullanılıyor.<br><br>CamScanner 2011'de piyasaya çıktığında tıklamadan belge tara özelliği devrim yarattı. Algoritmasının temelinde <code>cv.getPerspectiveTransform()</code> var. Şu an bu kütüphane 500 milyon+ kullanıcı tarafından günde milyarlarca kez çalıştırılıyor.</span></div>

## 2.3 Adım Adım Çözüm

<div style="background: #0d2200; border-left: 5px solid #90be6d; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #90be6d; font-size: 1.08em;">📍 Adım 1 </strong><br><br><span style="color: #cdd6f4; line-height: 1.9;">Sentetik eğri belge görüntüsü oluştur.</span></div>

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

def sentetik_belge_olustur(boyut=500):
    """Gerçekçi perspektif bozukluğu olan bir belge görüntüsü."""
    # Temiz belge
    belge = np.ones((400, 300, 3), dtype=np.uint8) * 245  # beyazımsı

    # Çizgili sayfa görünümü
    for y in range(30, 380, 22):
        cv.line(belge, (20, y), (280, y), (180, 180, 220), 1)

    # Metin simülasyonu
    for y in range(40, 370, 22):
        uzunluk = np.random.randint(100, 255)
        cv.rectangle(belge, (20, y-8), (20+uzunluk, y-2), (60, 60, 80), -1)

    # Başlık
    cv.putText(belge, "BELGE", (80, 25), cv.FONT_HERSHEY_DUPLEX, 0.8, (30,30,120), 2)
    cv.rectangle(belge, (5,5), (295,395), (100,100,150), 2)

    # Perspektif bozukluğu: 4 köşeyi eğ
    src_pts = np.float32([[0,0],[300,0],[300,400],[0,400]])
    dst_pts = np.float32([[40,20],[boyut-20,0],[boyut-40,boyut-30],[20,boyut-10]])
    M = cv.getPerspectiveTransform(src_pts, dst_pts)
    egri = cv.warpPerspective(belge, M, (boyut, boyut),
                              borderMode=cv.BORDER_CONSTANT,
                              borderValue=(100, 100, 100))
    return egri, belge


egri_belge, gercek_belge = sentetik_belge_olustur()
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(cv.cvtColor(egri_belge, cv.COLOR_BGR2RGB))
axes[0].set_title("Eğri Çekilmiş Belge (Giriş)", fontsize=10)
axes[0].axis("off")
axes[1].imshow(cv.cvtColor(gercek_belge, cv.COLOR_BGR2RGB))
axes[1].set_title("Gerçek Belge (Hedef)", fontsize=10)
axes[1].axis("off")
plt.tight_layout()
plt.show()

<div style="background: #0d2200; border-left: 5px solid #90be6d; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #90be6d; font-size: 1.08em;">📍 Adım 2 </strong><br><br><span style="color: #cdd6f4; line-height: 1.9;">Kenar tespiti ve kontur bulma.</span></div>

In [ ]:
def on_isle(goruntu):
    """
    Görüntüyü belge kenarı tespitine hazırlar.
    Gürültüyü azalt, kenarları belirginleştir.
    """
    # Gri tonlamaya çevir
    gri = cv.cvtColor(goruntu, cv.COLOR_BGR2GRAY)

    # Gaussian Blur: küçük gürültü noktalarını yumuşat
    # Kernel boyutu tek sayı olmalı, σ=0 → otomatik hesaplan
    bulanik = cv.GaussianBlur(gri, (5, 5), 0)

    # Canny kenar tespiti
    # threshold1=75: zayıf kenar eşiği
    # threshold2=200: güçlü kenar eşiği
    # Hysteresis: zayıf kenar güçlü kenara bağlıysa dahil et
    kenarlar = cv.Canny(bulanik, 75, 200)

    # Morfoloji: kesik kenarları birleştir (dilation)
    kernel = np.ones((3, 3), np.uint8)
    kenarlar = cv.dilate(kenarlar, kernel, iterations=1)

    return kenarlar


kenarlar = on_isle(egri_belge)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(cv.cvtColor(egri_belge, cv.COLOR_BGR2RGB))
axes[0].set_title("Orijinal")
axes[0].axis("off")
axes[1].imshow(cv.cvtColor(egri_belge, cv.COLOR_BGR2GRAY), cmap="gray")
axes[1].set_title("Gri Tonlama")
axes[1].axis("off")
axes[2].imshow(kenarlar, cmap="gray")
axes[2].set_title("Canny Kenar Tespiti + Dilate")
axes[2].axis("off")
plt.suptitle("Ön İşleme Adımları", fontsize=12)
plt.tight_layout()
plt.show()

<div style="background: #0d2200; border-left: 5px solid #90be6d; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #90be6d; font-size: 1.08em;">📍 Adım 3 </strong><br><br><span style="color: #cdd6f4; line-height: 1.9;">En büyük dörtgeni bul ve köşe noktalarını sırala.</span></div>

In [ ]:
def en_buyuk_dortgeni_bul(kenarlar):
    """
    Konturlardan en büyük 4 köşeli şekli (belge sınırı) bulur.
    """
    # Tüm konturları bul
    konturlar, _ = cv.findContours(kenarlar, cv.RETR_LIST, cv.CHAIN_APPROX_SIMPLE)

    # Alana göre büyükten küçüğe sırala, ilk 5'i incele
    konturlar = sorted(konturlar, key=cv.contourArea, reverse=True)[:5]

    belge_konturu = None
    for kontur in konturlar:
        # Kontur çevresi
        cevre = cv.arcLength(kontur, True)

        # Köşe sayısını yaklaştır: epsilon = %2 hassasiyet
        # True: kapalı kontur
        yaklasik = cv.approxPolyDP(kontur, 0.02 * cevre, True)

        if len(yaklasik) == 4:  # 4 köşe = dörtgen
            belge_konturu = yaklasik
            break

    return belge_konturu


def koseleri_sirala(noktalar):
    """
    4 köşeyi şu sıraya getirir: [sol-üst, sağ-üst, sağ-alt, sol-alt]
    Bu sıra getPerspectiveTransform için gerekli.
    """
    noktalar = noktalar.reshape(4, 2).astype(np.float32)

    sirali = np.zeros((4, 2), dtype=np.float32)

    # x+y toplamı: sol-üst en küçük, sağ-alt en büyük
    toplam = noktalar.sum(axis=1)
    sirali[0] = noktalar[np.argmin(toplam)]  # sol-üst
    sirali[2] = noktalar[np.argmax(toplam)]  # sağ-alt

    # x-y farkı: sağ-üst en küçük fark, sol-alt en büyük fark
    fark = np.diff(noktalar, axis=1)
    sirali[1] = noktalar[np.argmin(fark)]   # sağ-üst
    sirali[3] = noktalar[np.argmax(fark)]   # sol-alt

    return sirali


belge_konturu = en_buyuk_dortgeni_bul(kenarlar)

if belge_konturu is not None:
    print(f"Belge konturu bulundu! {len(belge_konturu)} köşe nokta.")
    print("Köşe koordinatları:\n", belge_konturu.reshape(4, 2))

    # Görselleştir
    gosterim = egri_belge.copy()
    cv.drawContours(gosterim, [belge_konturu], -1, (0, 255, 0), 3)
    for pnt in belge_konturu:
        cv.circle(gosterim, tuple(pnt[0]), 8, (0, 0, 255), -1)

    plt.figure(figsize=(5, 5))
    plt.imshow(cv.cvtColor(gosterim, cv.COLOR_BGR2RGB))
    plt.title("Tespit Edilen Belge Sınırı\n(Yeşil: kontur, Kırmızı: köşeler)", fontsize=9)
    plt.axis("off")
    plt.show()
else:
    print("Belge konturu bulunamadı — parametreleri ayarlayın")

<div style="background: #0d2200; border-left: 5px solid #90be6d; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #90be6d; font-size: 1.08em;">📍 Adım 4 </strong><br><br><span style="color: #cdd6f4; line-height: 1.9;">Perspektif dönüşümünü uygula ve belgeyi düzleştir.</span></div>

In [ ]:
def belge_tara(goruntu, kontur, hedef_genislik=595, hedef_yukseklik=842):
    """
    Perspektif dönüşümü uygulayarak belgeyi A4 formatına getirir.
    595×842: A4 @ 72 DPI (noktada)
    1240×1754: A4 @ 150 DPI (baskı kalitesi)
    2480×3508: A4 @ 300 DPI (profesyonel baskı)
    """
    # Kaynak: tespit edilen köşeler (sıralı)
    src = koseleri_sirala(kontur)

    # Hedef: dikdörtgen A4 boyutu
    dst = np.float32([
        [0,                 0],
        [hedef_genislik-1,  0],
        [hedef_genislik-1,  hedef_yukseklik-1],
        [0,                 hedef_yukseklik-1]
    ])

    # 3×3 perspektif matrisi (4 nokta → tam belirlenir)
    M = cv.getPerspectiveTransform(src, dst)
    print("Perspektif Matrisi:")
    print(np.round(M, 4))

    # Dönüşümü uygula
    taranmis = cv.warpPerspective(goruntu, M, (hedef_genislik, hedef_yukseklik))

    return taranmis


if belge_konturu is not None:
    taranmis = belge_tara(egri_belge, belge_konturu, 300, 400)

    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    axes[0].imshow(cv.cvtColor(egri_belge, cv.COLOR_BGR2RGB))
    axes[0].set_title("Giriş: Eğri Çekilmiş", fontsize=9)
    axes[0].axis("off")

    axes[1].imshow(cv.cvtColor(taranmis, cv.COLOR_BGR2RGB))
    axes[1].set_title("Çıkış: Taranan Belge\n(Perspektif Düzeltildi)", fontsize=9)
    axes[1].axis("off")

    axes[2].imshow(cv.cvtColor(gercek_belge, cv.COLOR_BGR2RGB))
    axes[2].set_title("Referans: Gerçek Belge", fontsize=9)
    axes[2].axis("off")

    plt.suptitle("Otomatik Belge Tarayıcı — Sonuç", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()

<div style="background: #001e2d; border-left: 5px solid #00b4d8; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #00b4d8; font-size: 1.08em;">🔬 Teknik Derinlik</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>getPerspectiveTransform: 8 bilinmeyenli denklem sistemi</b><br><br>Perspektif matrisi 3×3 = 9 eleman içerir ama ölçek serbesttir (homojen koordinatlar), dolayısıyla 8 serbestlik derecesi var. Her nokta çifti 2 denklem verir → 4 nokta = 8 denklem = tam belirlenir.<br><br>3 nokta yeterli olsaydı Affine dönüşüm yeterli olurdu. Perspektif için kesinlikle 4 nokta gereklidir. 5 nokta olursa sistem aşırı belirlenir → en küçük kareler yöntemi gerekir (<code>findHomography</code>).<br><br><b>Gerçek dünya zorluğu:</b> Gürültülü görüntülerde 4 köşe tespit etmek her zaman bu kadar kolay değil. CamScanner derin öğrenme tabanlı köşe dedektörleri kullanmaya geçti (DORN, DeepDoc algoritmaları).</span></div>

---
# 🏔️ SORU 3: Görüntü Piramidi — Gaussian & Laplacian

## 3.1 Neden Piramit?

Görüntü piramidi, aynı görüntünün farklı çözünürlükteki versiyonlarını hiyerarşik olarak saklar. Alt seviye: tam çözünürlük. Üst seviye: bulanık + küçük.

```
Seviye 4:  ■          (en küçük, en bulanık)
Seviye 3:  ■■■        
Seviye 2:  ■■■■■■■    
Seviye 1:  ■■■■■■■■■■■■■
Seviye 0:  (orijinal — en büyük, en detaylı)
```

**Kullanım alanları:**
- Çok ölçekli nesne tespiti (yüzler farklı boyutlarda olabilir)
- Görüntü harmanlama (doğal geçiş)
- JPEG/wavelet sıkıştırma
- Optik akış hesaplama (kaba-ince strateji)

<div style="background: #2d1800; border-left: 5px solid #f4a261; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #f4a261; font-size: 1.08em;">📜 Tarihçe</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>Piramit Fikrini Kim Buldu? — Burt & Adelson, 1983:</b><br><br>Peter Burt ve Edward Adelson, "The Laplacian Pyramid as a Compact Image Code" makalesini 1983'te yayımladı. Bu makale görüntü işlemede <b>en çok atıf alan çalışmalar</b> arasında yer alır.<br><br>Temel fikir: insan görsel korteksi de görüntüyü farklı frekans bantlarında (detay seviyelerinde) ayrı ayrı işler — V1 korteksinde farklı frekans bandlarına duyarlı hücreler bulunmuştur. Laplacian piramidi bu biyolojik mekanizmayı taklit eder.<br><br><b>JPEG bağlantısı:</b> JPEG'in kullandığı DCT (Ayrık Kosinüs Dönüşümü) ve wavelet tabanlı JPEG 2000, Laplacian piramidiyle aynı temel fikri paylaşır: görüntüyü frekans bileşenlerine ayır, önemsizleri at, önemlileri sakla.</span></div>

## 3.2 Adım Adım Çözüm

<div style="background: #0d2200; border-left: 5px solid #90be6d; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #90be6d; font-size: 1.08em;">📍 Adım 1 </strong><br><br><span style="color: #cdd6f4; line-height: 1.9;">Gaussian piramidini inşa et.</span></div>

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

# Test görüntüsü
img = np.zeros((256, 256, 3), dtype=np.uint8)
cv.circle(img, (128,128), 80, (200,80,80), -1)
cv.rectangle(img, (30,30), (120,120), (80,200,80), -1)
cv.putText(img, "PIRAMIT", (20, 240), cv.FONT_HERSHEY_SIMPLEX, 0.7, (255,220,0), 2)


def gaussian_piramit_insa(goruntu, seviye=5):
    """
    Gaussian piramidi: her adımda Gaussian bulanıklaştırma + yarıya indirme.
    cv.pyrDown() = GaussianBlur + subsample (her 2. piksel)
    
    Matematik:
      G[0] = orijinal
      G[k] = subsample(GaussianBlur(G[k-1]))
    """
    piramit = [goruntu.copy()]

    for k in range(seviye - 1):
        # pyrDown: önce 5×5 Gaussian filtre, sonra çift satır/sütunları at
        asagi = cv.pyrDown(piramit[-1])
        piramit.append(asagi)
        print(f"G[{k+1}] boyut: {asagi.shape[1]}×{asagi.shape[0]}")

    return piramit


gauss_pyr = gaussian_piramit_insa(img, seviye=5)

# Görselleştir
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
fig.suptitle("Gaussian Piramidi — Her Seviye Yarı Boyut", fontsize=12)
for i, (ax, katman) in enumerate(zip(axes, gauss_pyr)):
    ax.imshow(cv.cvtColor(katman, cv.COLOR_BGR2RGB))
    ax.set_title(f"G[{i}]\n{katman.shape[1]}×{katman.shape[0]}", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

<div style="background: #0d2200; border-left: 5px solid #90be6d; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #90be6d; font-size: 1.08em;">📍 Adım 2 </strong><br><br><span style="color: #cdd6f4; line-height: 1.9;">Laplacian piramidini inşa et.</span></div>

In [ ]:
def laplacian_piramit_insa(gauss_pyr):
    """
    Laplacian piramidi: Gaussian farkları.
    
    Matematik:
      L[k] = G[k] - pyrUp(G[k+1])
    
    L[k] = o seviyedeki 'detay' (yüksek frekans bileşeni)
    En üst seviye aynen saklanır (geri dönüşüm için gerekli).
    
    Neden 'Laplacian' ismi?
    Görüntü işlemede Laplacian operatörü kenar/detay tespit eder.
    Bu piramit katmanları da detayları (Gaussian farkı = band-pass filtre) içerir.
    """
    n = len(gauss_pyr)
    lap_pyr = []

    for k in range(n - 1):
        # Bir üst seviyeyi (daha küçük) büyüt
        buyutulmus = cv.pyrUp(gauss_pyr[k + 1])

        # Boyut uyumsuzluğunu düzelt (tek sayı boyutlarda 1 piksel fark olabilir)
        h, w = gauss_pyr[k].shape[:2]
        buyutulmus = cv.resize(buyutulmus, (w, h))

        # Farkı hesapla: detay bilgisi
        # float32'ye çevir — negatif değerler olabilir!
        fark = cv.subtract(gauss_pyr[k].astype(np.float32),
                           buyutulmus.astype(np.float32))
        lap_pyr.append(fark)

    # En üst Gaussian seviyesini olduğu gibi ekle
    lap_pyr.append(gauss_pyr[-1].astype(np.float32))

    return lap_pyr


lap_pyr = laplacian_piramit_insa(gauss_pyr)

# Görselleştir: detayları görmek için normalize et
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
fig.suptitle("Laplacian Piramidi — Her Katman = Detay Bilgisi", fontsize=12)
for i, (ax, katman) in enumerate(zip(axes, lap_pyr)):
    # Normalize: [-255,255] → [0,255] gösterim için
    gosterim = np.clip(katman + 128, 0, 255).astype(np.uint8)
    ax.imshow(cv.cvtColor(gosterim, cv.COLOR_BGR2RGB))
    ax.set_title(f"L[{i}]\n{katman.shape[1]}×{katman.shape[0]}", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

<div style="background: #0d2200; border-left: 5px solid #90be6d; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #90be6d; font-size: 1.08em;">📍 Adım 3 </strong><br><br><span style="color: #cdd6f4; line-height: 1.9;">Orijinal görüntüyü Laplacian piramidinden geri yap.</span></div>

In [ ]:
def piramidden_yeniden_yap(lap_pyr):
    """
    Laplacian piramidinden orijinal görüntüyü geri oluşturur.
    
    Matematik (tersi işlem):
      G[k] = L[k] + pyrUp(G[k+1])
    
    En üst seviyeden (en bulanık) başlayıp
    her adımda bir alt seviyenin detayını ekle.
    """
    # En üst Gaussian seviyesinden başla
    geri = lap_pyr[-1].copy()

    # Yukarıdan aşağıya inerek detayları ekle
    for k in range(len(lap_pyr) - 2, -1, -1):
        # Bir boyut büyüt
        buyutulmus = cv.pyrUp(geri)

        # Boyut uyumsuzluğunu düzelt
        h, w = lap_pyr[k].shape[:2]
        buyutulmus = cv.resize(buyutulmus, (w, h))

        # Detayı ekle
        geri = buyutulmus + lap_pyr[k]

    return np.clip(geri, 0, 255).astype(np.uint8)


yeniden_yapilmis = piramidden_yeniden_yap(lap_pyr)

# Kalite kontrolü
hata = cv.absdiff(img, yeniden_yapilmis)
mse = np.mean(hata.astype(float)**2)
print(f"Yeniden yapılandırma MSE: {mse:.6f} (0'a çok yakın = mükemmel)")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
axes[0].set_title("Orijinal")
axes[0].axis("off")
axes[1].imshow(cv.cvtColor(yeniden_yapilmis, cv.COLOR_BGR2RGB))
axes[1].set_title(f"Yeniden Yapılandırılmış\nMSE={mse:.6f}")
axes[1].axis("off")
axes[2].imshow(hata * 10)
axes[2].set_title("Hata ×10 (Neredeyse Sıfır)")
axes[2].axis("off")
plt.suptitle("Laplacian Piramidi → Geri Yapılandırma", fontsize=12)
plt.tight_layout()
plt.show()

<div style="background: #0d2200; border-left: 5px solid #90be6d; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #90be6d; font-size: 1.08em;">📍 Adım 4 </strong><br><br><span style="color: #cdd6f4; line-height: 1.9;">Piramit tabanlı görüntü harmanlama.</span></div>

In [ ]:
def piramit_harmonla(img1, img2, seviye=4):
    """
    İki görüntüyü Laplacian piramidi ile soldan sağa harmanlar.
    
    Basit kopyalama: sert kenar oluşur
    Piramit harmanlama: her frekans bandında ayrı harmanlama → doğal geçiş
    
    Temel fikir:
    - Düşük frekanslı bilgi (genel şekil): geniş geçiş bölgesi
    - Yüksek frekanslı bilgi (detaylar): dar/sert geçiş
    Bu insan görsel sisteminin algısına uyar.
    """
    assert img1.shape == img2.shape, "Görüntü boyutları eşit olmalı"
    h, w = img1.shape[:2]

    # Gaussian piramitleri
    gp1 = [img1.astype(np.float32)]
    gp2 = [img2.astype(np.float32)]
    for _ in range(seviye):
        gp1.append(cv.pyrDown(gp1[-1]))
        gp2.append(cv.pyrDown(gp2[-1]))

    # Laplacian piramitleri
    lp1, lp2 = [], []
    for k in range(seviye):
        up1 = cv.pyrUp(gp1[k+1])
        up2 = cv.pyrUp(gp2[k+1])
        ch, cw = gp1[k].shape[:2]
        lp1.append(gp1[k] - cv.resize(up1, (cw, ch)))
        lp2.append(gp2[k] - cv.resize(up2, (cw, ch)))
    lp1.append(gp1[-1])
    lp2.append(gp2[-1])

    # Her seviyede sol yarı img1, sağ yarı img2
    harmonli_pyr = []
    for k in range(seviye + 1):
        kh, kw = lp1[k].shape[:2]
        orta = kw // 2
        katman = np.hstack([lp1[k][:, :orta], lp2[k][:, orta:]])
        harmonli_pyr.append(katman)

    # Geri yap
    geri = harmonli_pyr[-1].copy()
    for k in range(seviye - 1, -1, -1):
        buyutulmus = cv.pyrUp(geri)
        kh, kw = harmonli_pyr[k].shape[:2]
        geri = cv.resize(buyutulmus, (kw, kh)) + harmonli_pyr[k]

    return np.clip(geri, 0, 255).astype(np.uint8)


# Test görüntüleri
elma = np.zeros((256, 256, 3), dtype=np.uint8)
cv.circle(elma, (128, 128), 100, (30, 30, 200), -1)
cv.circle(elma, (128, 128), 30, (150, 150, 255), -1)

portakal = np.zeros((256, 256, 3), dtype=np.uint8)
cv.circle(portakal, (128, 128), 100, (0, 140, 255), -1)
cv.rectangle(portakal, (90, 90), (166, 166), (0, 200, 200), 3)

# Basit kopyalama vs piramit harmanlama
orta = 128
basit = np.hstack([elma[:, :orta], portakal[:, orta:]])
piramit_sonuc = piramit_harmonla(elma, portakal, seviye=4)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, (g, b) in zip(axes, [
    (elma,         "Görüntü 1"),
    (portakal,     "Görüntü 2"),
    (basit,        "Basit Kopyalama\n(Sert Kenar)"),
    (piramit_sonuc,"Piramit Harmanlama\n(Doğal Geçiş)"),
]):
    ax.imshow(cv.cvtColor(g, cv.COLOR_BGR2RGB))
    ax.set_title(b, fontsize=9)
    ax.axis("off")
plt.suptitle("Laplacian Piramit Harmanlama", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

<div style="background: #0d1e3d; border-left: 5px solid #4a9eff; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #4a9eff; font-size: 1.08em;">💡 Anahtar Fikir</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>Neden piramit harmanlama daha doğal görünür?</b><br><br>Basit yapıştırma: birleşim noktasında her frekansta sert kesme olur → göz hemen fark eder.<br><br>Piramit harmanlama: <b>düşük frekanslarda geniş geçiş, yüksek frekanslarda dar geçiş</b>. Bu, doğadaki gölge geçişlerinin nasıl çalıştığına benzer. İnsan görsel sistemi bu geçişi "doğal" olarak algılar.<br><br>Aynı fikir Photoshop'un "Auto-Blend Layers" özelliğinde, panorama birleştirmede ve gerçek zamanlı yüz değiştirme (face swap) algoritmalarında kullanılır.</span></div>

---
# 🎥 SORU 4: Gerçek Zamanlı Video Stabilizasyon

## 4.1 Problem Analizi

El titremesi videoyu sarsar: her kare bir öncekine göre rastgele kaymış ve dönerek gelir. Stabilizasyon: bu rastgele hareketi tespit et, pürüzsüz bir hareket yoluna (smooth trajectory) otur, farkı ters dönüşümle iptal et.

## 4.2 Algoritma Adımları

```
Kare[0] → Kare[1]:  dx=+3, dy=-2, dθ=+0.5°   (titreme)
Kare[1] → Kare[2]:  dx=-1, dy=+4, dθ=-1.2°
Kare[2] → Kare[3]:  dx=+5, dy=-1, dθ=+0.3°
        ...

Kümülatif:  [+3, +2, +5, ...]  (sallanıyor)
Hareketli Ortalama(k=30): [+2, +2.1, +2.05, ...]  (pürüzsüz)
Düzeltme = Kümülatif - Pürüzsüz  (iptal edilecek titreme)
Her kareye ters dönüşüm uygula → stabil video
```

<div style="background: #2d1800; border-left: 5px solid #f4a261; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #f4a261; font-size: 1.08em;">📜 Tarihçe</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>Video Stabilizasyonun Tarihi:</b><br><br>İlk optik stabilizasyon sistemleri 1960'larda askeri helikopter kameraları için geliştirildi — fiziksel jiroskopu ile lens gimbals üzerinde hareket ederdi. Canon, 1995'te Image Stabilizer (IS) teknolojisini tüketici kameralarına taşıdı.<br><br>Elektronik/dijital stabilizasyon (EIS) 2000'lerin başında akıllı telefon kameralarıyla popülerleşti. Google'ın Pixel telefonlarındaki "Cinematic Pan" ve Apple'ın "Action Mode" özellikleri, bu haftaki algoritmaya benzer ilkelere dayanır.<br><br>Modern sistemler artık hibrit: OIS (fiziksel) + EIS (dijital) birlikte kullanılır. OIS yüksek frekanslı titremeyi, EIS düşük frekanslı sarsıntıyı önler.</span></div>

## 4.3 Adım Adım Çözüm

<div style="background: #0d2200; border-left: 5px solid #90be6d; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #90be6d; font-size: 1.08em;">📍 Adım 1 </strong><br><br><span style="color: #cdd6f4; line-height: 1.9;">Titrek sentetik video oluştur ve kare çiftleri arasındaki dönüşümü hesapla.</span></div>

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt


def sentetik_titrek_video_olustur(kare_sayisi=80, boyut=320):
    """
    Kameranın sarsıldığı bir videoyu simüle eder.
    Gerçek hareketi (smooth path) üzerine Gaussian titreme ekler.
    """
    np.random.seed(0)
    kareler = []

    # Arka plan: zengin desen (optik akış için iyi)
    arka_plan = np.zeros((boyut*2, boyut*2, 3), dtype=np.uint8)
    for i in range(0, boyut*2, 20):
        cv.line(arka_plan, (i, 0), (i, boyut*2), (60, 60, 80), 1)
        cv.line(arka_plan, (0, i), (boyut*2, i), (60, 60, 80), 1)
    for _ in range(30):
        cx = np.random.randint(20, boyut*2-20)
        cy = np.random.randint(20, boyut*2-20)
        r  = np.random.randint(10, 40)
        renk = tuple(np.random.randint(80, 220, 3).tolist())
        cv.circle(arka_plan, (cx, cy), r, renk, -1)
    cv.putText(arka_plan, "STABILIZASYON TESTI", (30, boyut),
               cv.FONT_HERSHEY_DUPLEX, 1.0, (220,200,50), 2)

    # Pürüzsüz hareket + titreme
    gercek_x = np.linspace(0, 100, kare_sayisi)
    gercek_y = 20 * np.sin(np.linspace(0, 2*np.pi, kare_sayisi))

    titreme_x = np.random.normal(0, 5, kare_sayisi)   # σ=5 piksel yatay titreme
    titreme_y = np.random.normal(0, 3, kare_sayisi)   # σ=3 piksel dikey titreme
    titreme_a = np.random.normal(0, 0.8, kare_sayisi) # σ=0.8° açısal titreme

    for i in range(kare_sayisi):
        x = gercek_x[i] + titreme_x[i]
        y = gercek_y[i] + titreme_y[i]
        a = titreme_a[i]

        # Affine dönüşüm: öteleme + hafif dönme
        M = np.float32([[np.cos(np.radians(a)), -np.sin(np.radians(a)), x + boyut//2],
                        [np.sin(np.radians(a)),  np.cos(np.radians(a)), y + boyut//2]])
        kare = cv.warpAffine(arka_plan, M, (boyut, boyut))
        kareler.append(kare)

    return kareler


kareler = sentetik_titrek_video_olustur(kare_sayisi=80)
print(f"{len(kareler)} kare oluşturuldu, boyut: {kareler[0].shape}")
plt.figure(figsize=(10, 3))
for i, idx in enumerate([0, 20, 40, 60]):
    plt.subplot(1, 4, i+1)
    plt.imshow(cv.cvtColor(kareler[idx], cv.COLOR_BGR2RGB))
    plt.title(f"Kare {idx}")
    plt.axis("off")
plt.suptitle("Titrek Video — Örnek Kareler", fontsize=10)
plt.tight_layout()
plt.show()

<div style="background: #0d2200; border-left: 5px solid #90be6d; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #90be6d; font-size: 1.08em;">📍 Adım 2 </strong><br><br><span style="color: #cdd6f4; line-height: 1.9;">Kare-kare dönüşümleri tahmin et (optik akış ile).</span></div>

In [ ]:
def donusumleri_tahmin_et(kareler):
    """
    Ardışık kare çiftleri arasındaki hareketi tahmin eder.
    
    Yöntem: estimateAffinePartial2D
    - Giriş: iki karede eşleşen nokta çiftleri (goodFeaturesToTrack + calcOpticalFlowPyrLK)
    - Çıkış: 2×3 affine matris (4 serbestlik: Δx, Δy, Δθ, Δscale)
    
    Alternatif: phase correlation (frekans alanında dönüşüm tahmini)
    """
    donusumler = []   # her kare için [dx, dy, da] listesi

    onceki = cv.cvtColor(kareler[0], cv.COLOR_BGR2GRAY)

    for i in range(1, len(kareler)):
        simdi = cv.cvtColor(kareler[i], cv.COLOR_BGR2GRAY)

        # Önceki karede iyi köşe noktaları bul (Shi-Tomasi)
        noktalar = cv.goodFeaturesToTrack(
            onceki,
            maxCorners=200,    # en fazla 200 nokta
            qualityLevel=0.01, # minimum kalite skoru
            minDistance=10     # noktalar arası minimum mesafe
        )

        if noktalar is None or len(noktalar) < 4:
            donusumler.append([0, 0, 0])
            onceki = simdi
            continue

        # Lucas-Kanade Optik Akış: noktaları takip et
        yeni_noktalar, durum, _ = cv.calcOpticalFlowPyrLK(
            onceki, simdi, noktalar, None
        )

        # Yalnızca başarıyla takip edilen noktaları kullan
        iyi_eski = noktalar[durum == 1]
        iyi_yeni = yeni_noktalar[durum == 1]

        # Affine dönüşüm matrisini tahmin et (RANSAC ile sağlam)
        M, _ = cv.estimateAffinePartial2D(iyi_eski, iyi_yeni)

        if M is None:
            donusumler.append([0, 0, 0])
        else:
            dx = M[0, 2]
            dy = M[1, 2]
            da = np.arctan2(M[1, 0], M[0, 0])  # radyan cinsinden açı
            donusumler.append([dx, dy, da])

        onceki = simdi

    return np.array(donusumler)


donusumler = donusumleri_tahmin_et(kareler)
print(f"Tahmin edilen dönüşümler: {donusumler.shape}")
print(f"Ortalama |dx|={np.mean(np.abs(donusumler[:,0])):.2f}, "
      f"|dy|={np.mean(np.abs(donusumler[:,1])):.2f}, "
      f"|da|={np.degrees(np.mean(np.abs(donusumler[:,2]))):.2f}°")

<div style="background: #0d2200; border-left: 5px solid #90be6d; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #90be6d; font-size: 1.08em;">📍 Adım 3 </strong><br><br><span style="color: #cdd6f4; line-height: 1.9;">Kümülatif hareket ve hareketli ortalama ile pürüzsüzleştir.</span></div>

In [ ]:
def hareketli_ortalama(dizi, pencere):
    """Kenar etkilerini azaltan kümülatif hareketli ortalama."""
    n = len(dizi)
    sonuc = np.zeros_like(dizi)
    for i in range(n):
        bas = max(0, i - pencere // 2)
        bitis = min(n, i + pencere // 2 + 1)
        sonuc[i] = np.mean(dizi[bas:bitis], axis=0)
    return sonuc


# Kümülatif yol: her dönüşümü bir öncekine ekle
kumulatif = np.cumsum(donusumler, axis=0)

# Hareketli ortalama: pürüzsüz hedef yol
PENCERE = 15  # kare sayısı — büyük pencere = daha pürüzsüz ama daha fazla kırpma
puruzsuz = hareketli_ortalama(kumulatif, PENCERE)

# Düzeltme: pürüzsüz yol - ham yol = uygulamamız gereken ters dönüşüm
duzeltme = puruzsuz - kumulatif

# Görselleştir
fig, axes = plt.subplots(3, 1, figsize=(13, 8))
for ax, (veri_ham, veri_pur, etiket) in zip(axes, [
    (kumulatif[:,0], puruzsuz[:,0], "Yatay (dx)"),
    (kumulatif[:,1], puruzsuz[:,1], "Dikey (dy)"),
    (np.degrees(kumulatif[:,2]), np.degrees(puruzsuz[:,2]), "Açı (°)"),
]):
    ax.plot(veri_ham, label="Ham (titrek)", alpha=0.6, color="#e94560")
    ax.plot(veri_pur, label=f"Pürüzsüz (pencere={PENCERE})", linewidth=2, color="#90be6d")
    ax.set_ylabel(etiket)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)
    ax.set_xlabel("Kare")
plt.suptitle("Kümülatif Kamera Hareketi — Ham vs Pürüzsüz", fontsize=12)
plt.tight_layout()
plt.show()

<div style="background: #0d2200; border-left: 5px solid #90be6d; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #90be6d; font-size: 1.08em;">📍 Adım 4 </strong><br><br><span style="color: #cdd6f4; line-height: 1.9;">Düzeltilmiş kareleri üret ve karşılaştır.</span></div>

In [ ]:
def stabilize_et(kareler, donusumler, pencere=15):
    """Tüm stabilizasyon pipeline'ını uygular."""
    kumulatif = np.cumsum(donusumler, axis=0)
    puruzsuz  = hareketli_ortalama(kumulatif, pencere)
    duzeltme  = puruzsuz - kumulatif

    stabilize_kareler = [kareler[0]]
    h, w = kareler[0].shape[:2]

    for i in range(1, len(kareler)):
        dx = duzeltme[i-1, 0]
        dy = duzeltme[i-1, 1]
        da = duzeltme[i-1, 2]

        # Düzeltme matrisini oluştur
        M = np.float32([
            [np.cos(da), -np.sin(da), dx],
            [np.sin(da),  np.cos(da), dy]
        ])

        # Uygula
        sabit_kare = cv.warpAffine(kareler[i], M, (w, h))
        stabilize_kareler.append(sabit_kare)

    return stabilize_kareler


stabilize = stabilize_et(kareler, donusumler, pencere=PENCERE)

# Yan yana karşılaştırma
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle("Video Stabilizasyon — Karşılaştırma", fontsize=12, fontweight="bold")

for col, kare_idx in enumerate([10, 25, 45, 65]):
    axes[0, col].imshow(cv.cvtColor(kareler[kare_idx], cv.COLOR_BGR2RGB))
    axes[0, col].set_title(f"Ham Kare {kare_idx}", fontsize=8)
    axes[0, col].axis("off")

    axes[1, col].imshow(cv.cvtColor(stabilize[kare_idx], cv.COLOR_BGR2RGB))
    axes[1, col].set_title(f"Stabilize Kare {kare_idx}", fontsize=8)
    axes[1, col].axis("off")

axes[0, 0].set_ylabel("Ham", fontsize=9)
axes[1, 0].set_ylabel("Stabilize", fontsize=9)
plt.tight_layout()
plt.show()

# Titreme miktarı karşılaştırması
ham_std = np.std(donusumler[:, :2])
stab_duz = hareketli_ortalama(np.cumsum(donusumler, axis=0), PENCERE)
kalan_titreme = np.cumsum(donusumler, axis=0) - stab_duz
stab_std = np.std(kalan_titreme[:, :2])
print(f"Ham titreme std: {ham_std:.2f} piksel")
print(f"Stabilize sonrası std: {stab_std:.2f} piksel")
print(f"Azalma: %{(1 - stab_std/ham_std)*100:.1f}")

---
# 🤖 SORU 5: Bounding Box Dahil Augmentation Pipeline

## 5.1 Neden Bounding Box Augmentasyonu Zordur?

Görüntüyü döndürünce bounding box koordinatları da dönmeli. Ama dikdörtgen bir kutu döndürülünce artık dikdörtgen olmayabilir! Bu durumda **axis-aligned bounding box** (eksene hizalı) kullanmak için döndürülmüş kutunun **dışını saran yeni dikdörtgeni** hesaplamak gerekir.

<div style="background: #1a0d2e; border-left: 5px solid #c77dff; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #c77dff; font-size: 1.08em;">🧠 Augmentation ve Derin Öğrenme İlişkisi</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>AlexNet (2012):</b> ImageNet şampiyonluğunu kazanan ilk derin ağ. Başarısının önemli bir nedeni agresif data augmentation'dı: yatay flip, rastgele kırpma, PCA renk bozma. Sadece augmentation ile doğrulama hatası %1.2 düştü.<br><br><b>Albumentations (2018):</b> Kaggle yarışmalarından çıkan, bounding box destekli augmentation için en hızlı kütüphane. OpenCV'yi backend olarak kullanır, 70+ augmentation türü içerir. Test zamanı bir ImageNet görüntüsüne tüm augmentasyonları uygulamak torchvision'dan 3-4× daha hızlıdır.<br><br><b>Kötü augmentation ne yapar?</b> Tıbbi görüntülemede yanlış augmentation felakete yol açabilir: akciğer grafisini dikey flip yaparsanız kalp sağda görünür — bu patolojik bir durum! Model "kalp sağda = normal" öğrenir.</span></div>

<div style="background: #0d2200; border-left: 5px solid #90be6d; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #90be6d; font-size: 1.08em;">📍 Adım 1 </strong><br><br><span style="color: #cdd6f4; line-height: 1.9;">Veri yapılarını tanımla.</span></div>

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from dataclasses import dataclass, field
from typing import List, Optional, Tuple


@dataclass
class BoundingBox:
    """
    Normalize koordinatlı bounding box (0.0 - 1.0 arası).
    Normalize kullanmak: görüntü boyutundan bağımsız, ölçek değişiminde otomatik doğru.
    """
    x1: float  # sol kenar
    y1: float  # üst kenar
    x2: float  # sağ kenar
    y2: float  # alt kenar
    sinif_id: int = 0
    sinif_adi: str = "nesne"

    def piksel_koordinatlar(self, w, h):
        """Normalize → piksel koordinatlara çevir."""
        return (int(self.x1*w), int(self.y1*h),
                int(self.x2*w), int(self.y2*h))

    def dortgen_koseleri(self, w, h):
        """4 köşe noktası döndürür (döndürme için)."""
        x1, y1, x2, y2 = self.piksel_koordinatlar(w, h)
        return np.float32([[x1,y1],[x2,y1],[x2,y2],[x1,y2]])


def goruntu_ciz(img, kutular, baslik=""):
    """Görüntü üzerine bounding box'ları çizer."""
    gosterim = img.copy()
    h, w = img.shape[:2]
    renkler = [(255,50,50),(50,255,50),(50,50,255),(255,255,0),(255,0,255)]
    for k, bb in enumerate(kutular):
        renk = renkler[k % len(renkler)]
        x1,y1,x2,y2 = bb.piksel_koordinatlar(w, h)
        cv.rectangle(gosterim, (x1,y1), (x2,y2), renk, 2)
        cv.putText(gosterim, bb.sinif_adi, (x1, y1-5),
                   cv.FONT_HERSHEY_SIMPLEX, 0.45, renk, 1)
    return gosterim


print("Veri yapıları hazır.")

<div style="background: #0d2200; border-left: 5px solid #90be6d; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #90be6d; font-size: 1.08em;">📍 Adım 2 </strong><br><br><span style="color: #cdd6f4; line-height: 1.9;">Her augmentation fonksiyonunu uygula.</span></div>

In [ ]:
class AugmentationPipeline:
    """
    Görüntü ve bounding box'ları birlikte dönüştüren augmentation pipeline'ı.
    
    Tüm işlemler aynı rastgele seed ile yapılır:
    görüntü ve etiketler her zaman senkron kalır.
    """

    def __init__(self, seed: Optional[int] = None):
        self.rng = np.random.default_rng(seed)

    # ─── Yatay Çevirme ───────────────────────────────────────────────────
    def yatay_flip(self, img, kutular):
        """
        Görüntüyü ve bounding box'ları sağ-sol çevirir.
        x koordinatları: x_yeni = 1 - x_eski
        """
        img_flip = cv.flip(img, 1)  # flipCode=1: yatay eksen

        yeni_kutular = []
        for bb in kutular:
            # Sol-sağ koordinatlar yer değiştirir
            yeni = BoundingBox(
                x1=1.0 - bb.x2,  # eski sağ → yeni sol
                y1=bb.y1,
                x2=1.0 - bb.x1,  # eski sol → yeni sağ
                y2=bb.y2,
                sinif_id=bb.sinif_id,
                sinif_adi=bb.sinif_adi
            )
            yeni_kutular.append(yeni)

        return img_flip, yeni_kutular

    # ─── Döndürme ─────────────────────────────────────────────────────────
    def dondur(self, img, kutular, aci):
        """
        Görüntüyü merkezi etrafında döndürür ve bounding box'ları günceller.
        
        Bounding box döndürme stratejisi:
        1. 4 köşeyi al
        2. Her köşeye rotasyon matrisini uygula
        3. Döndürülmüş köşelerin axis-aligned (eksene hizalı) bounding box'ını hesapla
           (yani min/max x ve y değerlerini al)
        """
        h, w = img.shape[:2]
        cx, cy = w / 2, h / 2
        M = cv.getRotationMatrix2D((cx, cy), aci, 1.0)
        img_rot = cv.warpAffine(img, M, (w, h), flags=cv.INTER_LINEAR,
                                borderMode=cv.BORDER_REFLECT_101)

        yeni_kutular = []
        for bb in kutular:
            # 4 köşeyi piksel koordinatlara çevir
            koseler = bb.dortgen_koseleri(w, h)
            # Homojen koordinata çevir (N×3)
            koseler_h = np.hstack([koseler, np.ones((4,1))])
            # Rotasyon matrisini uygula
            donmus = (M @ koseler_h.T).T  # (4×2)

            # Axis-aligned bounding box: min/max değerler
            x_min = np.clip(donmus[:,0].min() / w, 0, 1)
            y_min = np.clip(donmus[:,1].min() / h, 0, 1)
            x_max = np.clip(donmus[:,0].max() / w, 0, 1)
            y_max = np.clip(donmus[:,1].max() / h, 0, 1)

            # Geçerli kutu mu? (çok küçük veya görüntü dışı değil mi)
            if x_max - x_min > 0.01 and y_max - y_min > 0.01:
                yeni_kutular.append(BoundingBox(
                    x1=x_min, y1=y_min, x2=x_max, y2=y_max,
                    sinif_id=bb.sinif_id, sinif_adi=bb.sinif_adi
                ))

        return img_rot, yeni_kutular

    # ─── Rastgele Kırpma ──────────────────────────────────────────────────
    def rastgele_kirp(self, img, kutular, oran_min=0.75, oran_max=1.0):
        """
        Görüntünün rastgele bir bölgesini kırpar.
        Bounding box'lar yeni koordinat sistemine taşınır.
        """
        h, w = img.shape[:2]
        oran = self.rng.uniform(oran_min, oran_max)
        yeni_h = int(h * oran)
        yeni_w = int(w * oran)

        y0 = self.rng.integers(0, h - yeni_h + 1)
        x0 = self.rng.integers(0, w - yeni_w + 1)

        kirpilmis = img[y0:y0+yeni_h, x0:x0+yeni_w]
        kirpilmis = cv.resize(kirpilmis, (w, h), interpolation=cv.INTER_LINEAR)

        # Koordinat dönüşümü: kırpma sonrası normalize
        x0_n = x0 / w
        y0_n = y0 / h
        w_n  = yeni_w / w
        h_n  = yeni_h / h

        yeni_kutular = []
        for bb in kutular:
            # Kırpma bölgesine göre yeni koordinatlar
            nx1 = (bb.x1 - x0_n) / w_n
            ny1 = (bb.y1 - y0_n) / h_n
            nx2 = (bb.x2 - x0_n) / w_n
            ny2 = (bb.y2 - y0_n) / h_n

            # Görüntü içinde kalan bölgeyi clip et
            nx1 = np.clip(nx1, 0, 1)
            ny1 = np.clip(ny1, 0, 1)
            nx2 = np.clip(nx2, 0, 1)
            ny2 = np.clip(ny2, 0, 1)

            # Görünür nesne yüzeyi yeterli mi? (>%20)
            orijinal_alan = (bb.x2 - bb.x1) * (bb.y2 - bb.y1)
            kalan_alan    = (nx2 - nx1) * (ny2 - ny1)
            if orijinal_alan > 0 and kalan_alan / orijinal_alan > 0.2:
                yeni_kutular.append(BoundingBox(
                    nx1, ny1, nx2, ny2,
                    bb.sinif_id, bb.sinif_adi
                ))

        return kirpilmis, yeni_kutular

    # ─── Parlaklık & Kontrast ─────────────────────────────────────────────
    def parlaklik_kontrast(self, img, kutular,
                           parlaklik_aralik=(-40, 40),
                           kontrast_aralik=(0.7, 1.3)):
        """
        Parlaklık ve kontrast değişimi.
        Bounding box'ları etkilemez (yalnızca piksel değerleri değişir).
        
        Formül: yeni_piksel = alpha × piksel + beta
          alpha: kontrast katsayısı
          beta:  parlaklık farkı
        """
        beta  = self.rng.integers(*parlaklik_aralik)
        alpha = self.rng.uniform(*kontrast_aralik)
        ayarli = cv.convertScaleAbs(img, alpha=alpha, beta=beta)
        return ayarli, kutular  # kutular değişmez!

    # ─── Gaussian Gürültü ─────────────────────────────────────────────────
    def gaussian_gurultu(self, img, kutular, sigma_max=12):
        """
        Kameradan gelen sensör gürültüsünü simüle eder.
        Bounding box'ları etkilemez.
        """
        sigma = self.rng.uniform(0, sigma_max)
        gurultu = self.rng.normal(0, sigma, img.shape).astype(np.int16)
        gurultulu = np.clip(img.astype(np.int16) + gurultu, 0, 255).astype(np.uint8)
        return gurultulu, kutular

    # ─── Cutout ──────────────────────────────────────────────────────────
    def cutout(self, img, kutular, oran=0.15):
        """
        DeVries & Taylor (2017) — Görüntünün rastgele bir bölgesini siyaha boyar.
        Modelin eksik veriyle başa çıkmasını öğretir (regularization).
        Bounding box'ları etkilemez (etiket değişmez).
        """
        h, w = img.shape[:2]
        bh = int(h * oran)
        bw = int(w * oran)
        y0 = self.rng.integers(0, h - bh + 1)
        x0 = self.rng.integers(0, w - bw + 1)

        kesik = img.copy()
        kesik[y0:y0+bh, x0:x0+bw] = 0  # siyah dikdörtgen
        return kesik, kutular

    # ─── Tam Pipeline ────────────────────────────────────────────────────
    def uygula(self, img, kutular: List[BoundingBox]) -> Tuple:
        """
        Tüm augmentasyonları belirli olasılıklarla uygular.
        Her çağrıda farklı kombinasyon → çeşitli öğrenme örnekleri.
        """
        # Yatay flip: %50 olasılık
        if self.rng.random() < 0.5:
            img, kutular = self.yatay_flip(img, kutular)

        # Döndürme: %60 olasılık, ±20°
        if self.rng.random() < 0.6:
            aci = float(self.rng.uniform(-20, 20))
            img, kutular = self.dondur(img, kutular, aci)

        # Kırpma: %50 olasılık
        if self.rng.random() < 0.5:
            img, kutular = self.rastgele_kirp(img, kutular)

        # Parlaklık/kontrast: %70 olasılık
        if self.rng.random() < 0.7:
            img, kutular = self.parlaklik_kontrast(img, kutular)

        # Gürültü: %40 olasılık
        if self.rng.random() < 0.4:
            img, kutular = self.gaussian_gurultu(img, kutular)

        # Cutout: %30 olasılık
        if self.rng.random() < 0.3:
            img, kutular = self.cutout(img, kutular)

        return img, kutular


print("AugmentationPipeline hazır.")

<div style="background: #0d2200; border-left: 5px solid #90be6d; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #90be6d; font-size: 1.08em;">📍 Adım 3 </strong><br><br><span style="color: #cdd6f4; line-height: 1.9;">Pipeline'ı test et ve görselleştir.</span></div>

In [ ]:
# ─── Test görüntüsü ve gerçekçi bounding box'lar ───
img_test = np.ones((300, 400, 3), dtype=np.uint8) * 180
# Araba
cv.rectangle(img_test, (30, 100), (180, 250), (70, 70, 200), -1)
cv.rectangle(img_test, (50, 60), (160, 110), (50, 50, 150), -1)
cv.circle(img_test, (70, 250), 25, (30,30,30), -1)
cv.circle(img_test, (150, 250), 25, (30,30,30), -1)
# İnsan
cv.rectangle(img_test, (230, 80), (300, 270), (200, 150, 100), -1)
cv.circle(img_test, (265, 55), 28, (220, 180, 140), -1)
# Ağaç
cv.rectangle(img_test, (330, 150), (375, 280), (60, 100, 40), -1)
cv.ellipse(img_test, (352, 110), (40, 50), 0, 0, 360, (40, 160, 40), -1)

# Bounding box'lar (normalize)
kutular_test = [
    BoundingBox(0.05, 0.20, 0.47, 0.88, sinif_id=0, sinif_adi="araba"),
    BoundingBox(0.56, 0.17, 0.77, 0.93, sinif_id=1, sinif_adi="insan"),
    BoundingBox(0.81, 0.17, 1.00, 0.97, sinif_id=2, sinif_adi="agac"),
]

# Pipeline'ı farklı seed'lerle çalıştır → her seferinde farklı augmentation
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
fig.suptitle("Augmentation Pipeline — Her Satır Farklı Augmentation Kombinasyonu\n"
             "Kırmızı: araba, Yeşil: insan, Mavi: ağaç", fontsize=11, fontweight="bold")

orijinal_gosterim = goruntu_ciz(img_test, kutular_test)
axes[0, 0].imshow(cv.cvtColor(orijinal_gosterim, cv.COLOR_BGR2RGB))
axes[0, 0].set_title("Orijinal", fontsize=9, fontweight="bold")
axes[0, 0].axis("off")

tekil_aug_aciklamalar = [
    ("Yatay Flip", lambda p,i,k: p.yatay_flip(i,k)),
    ("Döndürme +15°", lambda p,i,k: p.dondur(i,k,15)),
    ("Kırpma %80", lambda p,i,k: p.rastgele_kirp(i,k,0.8,0.8)),
]
for col, (aciklama, fonk) in enumerate(tekil_aug_aciklamalar, start=1):
    aug_img, aug_k = fonk(AugmentationPipeline(seed=42), img_test.copy(), list(kutular_test))
    gosterim = goruntu_ciz(aug_img, aug_k)
    axes[0, col].imshow(cv.cvtColor(gosterim, cv.COLOR_BGR2RGB))
    axes[0, col].set_title(aciklama, fontsize=8)
    axes[0, col].axis("off")

for satir in range(1, 3):
    for col in range(4):
        seed = satir * 10 + col
        pipeline = AugmentationPipeline(seed=seed)
        aug_img, aug_k = pipeline.uygula(img_test.copy(), list(kutular_test))
        gosterim = goruntu_ciz(aug_img, aug_k)
        axes[satir, col].imshow(cv.cvtColor(gosterim, cv.COLOR_BGR2RGB))
        axes[satir, col].set_title(f"Tam Pipeline #{seed}", fontsize=8)
        axes[satir, col].axis("off")

plt.tight_layout()
plt.show()
print("Augmentation pipeline demo tamamlandı.")

<div style="background: #0d1e3d; border-left: 5px solid #4a9eff; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #4a9eff; font-size: 1.08em;">💡 Anahtar Fikir</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>Augmentation ne kadar yapılmalı?</b><br><br>Kural yok — deneysel belirlenir. Ama genel rehber:<br>• Az veri (< 1000 örnek) → agresif augmentation<br>• Çok veri (> 100.000 örnek) → hafif augmentation<br>• Özelleşmiş alan (tıbbi, uydu) → domain-specific augmentation<br><br><b>Önemli uyarı:</b> Augmentation yalnızca <b>eğitim verisine</b> uygulanır! Test/validasyon verisi asla augmentation görmemeli — aksi hâlde modeli gerçek performansından farklı değerlendirirsiniz.</span></div>

---
<div style="background: linear-gradient(135deg, #0a0a0a 0%, #0d1e3d 50%, #1a0d2e 100%);padding: 38px 42px; border-radius: 18px; text-align: center; margin-top: 20px; border: 1px solid #4a9eff44;"><h2 style="color: #4a9eff; font-family: 'Courier New', monospace; margin: 0 0 22px 0;">✅ Çözümler Özeti</h2><div style="color: #cdd6f4; text-align: left; max-width: 700px; margin: 0 auto; line-height: 2.3;"><b>Soru 1 — SSIM/PSNR:</b> MSE piksel fark ölçer; PSNR dB cinsinden, SSIM yapısal benzerliği ölçer. Lanczos en yüksek SSIM. Wang & Bovik 2004 devrim yarattı.<br><b>Soru 2 — Belge Tarayıcı:</b> Canny → kontur → 4 köşe → köşeleri sırala → perspektif dönüşüm. Affine 3, perspektif 4 nokta ister; paralel çizgiler yalnızca affine'de korunur.<br><b>Soru 3 — Laplacian Piramidi:</b> G[k] = pyrDown(G[k-1]); L[k] = G[k] − pyrUp(G[k+1]). Burt & Adelson 1983. Harmanlama: her frekansta ayrı geçiş → doğal sonuç.<br><b>Soru 4 — Video Stabilizasyon:</b> goodFeaturesToTrack → optik akış → kümülatif → hareketli ortalama → ters warpAffine. OIS + EIS hibrit modern çözüm.<br><b>Soru 5 — Augmentation:</b> Her dönüşümde bbox koordinatları da güncellenmeli. Döndürmede 4 köşeyi döndür, axis-aligned bbox al. AlexNet'ten bu yana augmentation derin öğrenmenin vazgeçilmezi.</div></div>